# 结构化输出

In [1]:
from typing import Optional

from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
'''结构化输出：使用本地 Ollama 的 JSON Schema 模式'''
class Person(BaseModel):
    name: str=Field(description="姓名",min_length=2,max_length=15)
    age: Optional[int] = Field(description="年龄")
    occupation:str = Field(description="职业")
    sex:str=Field("男",description="性别")


model = ChatOpenAI(
    model="qwen3.5:9b-q4_K_M",
    base_url="http://127.0.0.1:11434/v1",
    api_key="ollama",
    temperature=0,
    # 关闭思考模式，直接生成结构化结果。
    reasoning_effort="none",
    timeout=120,
    max_retries=0,
)

structured_model = model.with_structured_output(Person, method="json_schema")

result = structured_model.invoke("张三是一名30岁工程师")

print(result)

name='张三' age=30 occupation='工程师' sex='男'


## 枚举类型

In [2]:
from enum import Enum
from pydantic import BaseModel, Field
'''枚举类型'''
class Priority(Enum):
    HIGH = "high"
    MEDIUM = "medium"
    LOW = "low"

class Task(BaseModel):
    task_id: str = Field(description="任务ID")
    priotity: Priority = Field(description="任务优先级")
    description: str = Field(description="任务描述")



## 列表提取
**注：因为大模型能力有限，复杂嵌套容易出错，建议嵌套层数<=3层**

In [3]:
from typing import List
from pydantic import BaseModel, Field
'''列表提取'''

class Actor(BaseModel):
    name: str = Field(description="演员姓名")
    role: str = Field(description="角色名称")

class Movie(BaseModel):
    title: str = Field(description="电影名称")
    year: int = Field(description="电影年份")
    director: str = Field(description="导演名称")
    cast: List[Actor] = Field(description="演员列表")
    rating: float = Field(description="电影评分")


structured_model = model.with_structured_output(Movie, method="json_schema")

result = structured_model.invoke("请介绍电影《肖申克的救赎》")

print(f"电影名称：{result.title}")
print(f"电影年份：{result.year}")
print(f"导演名称：{result.director}")
print(f"电影评分：{result.rating}")
print(f"演员列表：{result.cast}")


电影名称：肖申克的救赎
电影年份：1994
导演名称：弗兰克·德拉邦特
电影评分：9.3
演员列表：[Actor(name='蒂姆·罗宾斯', role='安迪·杜佛兰'), Actor(name='摩根·弗里曼', role='瑞德')]


In [4]:
from typing import List
from pydantic import BaseModel, Field

class Aspect(BaseModel):
    name: str = Field(description="维度名称，如：质量、价格、服务等")
    rating: int = Field(description="维度评分，1-5分")
    comment:str=Field(description="具体评论")


class ProductReview(BaseModel):
    '''产品评论分析'''
    overall_sentiment: str = Field(description="总体评价，如：好评、差评等")
    overall_score: int = Field(description="总体评分，1-5分")
    aspects: List[Aspect] = Field(description="评论维度列表")
    summary: str = Field(description="一句话总结")

structured_model = model.with_structured_output(ProductReview, method="json_schema")

review_text="这款手机质量很好，价格也合理，服务态度也很好。但是，电池续航时间较短。总体来说，我非常满意。"

result = structured_model.invoke(f"分析以下产品评论{review_text}")

print(f"总体评价：{result.overall_sentiment}")
print(f"总体评分：{result.overall_score}/5")
for aspect in result.aspects:
    print(f"-{aspect.name}:{aspect.rating}/5 - {aspect.comment}")

print(f"一句话总结：{result.summary}")





总体评价：positive
总体评分：4/5
-质量:5/5 - 很好
-价格:4/5 - 合理
-服务态度:5/5 - 很好
-电池续航:2/5 - 较短
一句话总结：用户对该手机的整体满意度较高，特别称赞了产品质量、价格合理性及服务态度，但指出电池续航时间存在不足。尽管有这一缺点，用户仍表示总体非常满意。
